# Dimension Customers


In [0]:
#Imports
from pyspark.sql.functions import col
from delta.tables import DeltaTable

In [0]:
customers_df = spark.read.table("neo_bank.silver.customers")
branches_df = spark.read.table("neo_bank.gold.dim_branches")

In [0]:
dim_customers_df = (
    customers_df.alias("c")
        .join(
            branches_df.alias("b"),
            col("c.branch_code")==col("b.branch_code"),
            "left"
        )
        .select(
            col("c.customer_id"),
            col("c.first_name"),
            col("c.last_name"),
            col("c.date_of_birth"),
            col("c.pan_number"),
            col("c.email"),
            col("c.phone_number"),
            col("c.kyc_status"),
            col("b.branch_sk").alias("home_branch_sk")
        )
)


In [0]:
dim_table = DeltaTable.forName(spark, "neo_bank.gold.dim_customers")

(
    dim_table.alias("target")
    .merge(
        dim_customers_df.alias("source"),
        condition="target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "first_name":"source.first_name",
        "last_name":"source.last_name",
        "date_of_birth":"source.date_of_birth",
        "pan_number":"source.pan_number",
        "email":"source.email",
        "phone_number":"source.phone_number",
        "kyc_status":"source.kyc_status",
        "home_branch_sk":"source.home_branch_sk"
    })
    .whenNotMatchedInsert(values={
        "customer_id":"source.customer_id",
        "first_name":"source.first_name",
        "last_name":"source.last_name",
        "date_of_birth":"source.date_of_birth",
        "pan_number":"source.pan_number",
        "email":"source.email",
        "phone_number":"source.phone_number",
        "kyc_status":"source.kyc_status",
        "home_branch_sk":"source.home_branch_sk"

    })
    .execute()
)